# Diffusion equations

Inspect a forward noise sample and a deterministic reverse step without a U-Net.

In [ ]:
from pathlib import Path
import importlib.util
lesson_rel = Path('phases/04-computer-vision/10-image-generation-diffusion')
candidates = []
for start in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(start / lesson_rel / 'code/main.py')
code_path = next(path.resolve() for path in candidates if path.is_file())
spec = importlib.util.spec_from_file_location('cv04_l10', code_path)
diffusion = importlib.util.module_from_spec(spec)
spec.loader.exec_module(diffusion)
print(code_path)

In [ ]:
import numpy as np
schedule = diffusion.precompute_schedule(diffusion.linear_beta_schedule(10, 1e-3, 0.02))
x0 = np.zeros((1, 1, 4, 4))
noise = np.ones_like(x0)
x_t = diffusion.q_sample(x0, 5, noise, schedule)
print('alpha_bar', schedule['alpha_bar'][5], 'x_t mean', float(x_t.mean()))

In [ ]:
recovered = diffusion.predict_x0_from_eps(x_t, 5, noise, schedule)
step = diffusion.ddim_step(x_t, 5, 2, noise, schedule, eta=0)
print('reconstruction error', float(np.abs(recovered - x0).max()), 'step shape', step.shape)
try:
    diffusion.ddim_step(x_t, 5, 2, noise, schedule, eta=0.1)
except ValueError as exc:
    print('eta contract:', exc)